# 03 — Kalibrasi threshold dan evaluasi open-set

Salinan publik dari alur eksperimen. Tidak memuat dataset, nama peserta, embedding, foto, output cell, atau path komputer penulis. Gunakan dataset sendiri yang berhak diproses. Hasil numerik akan bergantung pada dataset dan tidak harus sama dengan README.

Setelah dijalankan pada data pribadi, output notebook dapat kembali mengandung nama atau wajah. Jalankan script pembersih sebelum commit. Lihat `notebooks_public/README.md` untuk prasyarat dan batas reproduksi.

Selesaikan notebook 02 dahulu. Default menyisihkan 5 identitas unknown validasi dan 5 identitas unknown test; sisanya masuk galeri. Gunakan minimal 12 identitas pada konfigurasi default. Untuk dataset lebih kecil, ubah jumlah unknown sehingga sedikitnya dua identitas tetap menjadi known. Known dan unknown dibagi berdasarkan identitas, bukan hanya foto.

## Pembagian data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"

with np.load(
    ARTIFACTS_DIR / "arcface_embeddings.npz",
    allow_pickle=False
) as saved:
    X = saved["embeddings"]
    image_ids = saved["image_ids"]
    person_ids = saved["person_ids"]

classes = np.unique(person_ids)

N_VALIDATION_UNKNOWN = 5
N_TEST_UNKNOWN = 5
if N_VALIDATION_UNKNOWN < 1 or N_TEST_UNKNOWN < 1:
    raise ValueError("Kedua kelompok unknown memerlukan minimal satu identitas.")
n_known = len(classes) - N_VALIDATION_UNKNOWN - N_TEST_UNKNOWN
if n_known < 2:
    raise ValueError("Kurangi jumlah unknown atau tambahkan data; dibutuhkan minimal dua identitas known.")

if len(np.unique(image_ids)) != len(image_ids):
    raise ValueError("Ada image_id duplikat.")

counts = pd.Series(person_ids).value_counts()

if not counts.isin([3, 4]).all():
    raise ValueError("Periksa kembali jumlah foto per identitas.")

rng = np.random.default_rng(2026)
shuffled_classes = rng.permutation(classes)

known_ids = set(shuffled_classes[:n_known])
validation_unknown_ids = set(shuffled_classes[n_known:n_known + N_VALIDATION_UNKNOWN])
test_unknown_ids = set(shuffled_classes[n_known + N_VALIDATION_UNKNOWN:])

split_rows = []

for person_id in classes:
    indices = rng.permutation(
        np.flatnonzero(person_ids == person_id)
    )

    assignments = []

    if person_id in known_ids:
        # Dua foto untuk galeri.
        assignments.extend(
            (index, "gallery") for index in indices[:2]
        )

        remaining = indices[2:]

        if len(remaining) == 2:
            assignments.append(
                (remaining[0], "validation_known")
            )
            assignments.append(
                (remaining[1], "test_known")
            )
        else:
            # Identitas dengan tiga foto tetap dapat diuji.
            assignments.append(
                (remaining[0], "test_known")
            )

    elif person_id in validation_unknown_ids:
        assignments.extend(
            (index, "validation_unknown")
            for index in indices
        )

    else:
        assignments.extend(
            (index, "test_unknown")
            for index in indices
        )

    for index, role in assignments:
        split_rows.append({
            "image_id": image_ids[index],
            "person_id": person_id,
            "role": role,
        })

open_set_manifest_df = pd.DataFrame(split_rows)

if (
    len(open_set_manifest_df) != len(image_ids)
    or open_set_manifest_df["image_id"].duplicated().any()
):
    raise ValueError("Pembagian foto tidak lengkap atau tumpang tindih.")

manifest_path = METADATA_DIR / "open_set_manifest.csv"

open_set_manifest_df.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

role_order = [
    "gallery",
    "validation_known",
    "validation_unknown",
    "test_known",
    "test_unknown",
]

split_summary_df = (
    open_set_manifest_df
    .groupby("role")
    .agg(
        n_images=("image_id", "size"),
        n_identities=("person_id", "nunique")
    )
    .reindex(role_order)
    .reset_index()
)

display(split_summary_df)

print("Total foto:", len(open_set_manifest_df))
print("Manifest tersimpan:", manifest_path)

## Galeri centroid

In [ ]:
import numpy as np
import pandas as pd

embedding_path = ARTIFACTS_DIR / "arcface_embeddings.npz"
split_path = METADATA_DIR / "open_set_manifest.csv"

with np.load(embedding_path, allow_pickle=False) as saved:
    X = saved["embeddings"]
    image_ids = saved["image_ids"]
    person_ids = saved["person_ids"]

split_df = pd.read_csv(split_path)

if split_df["image_id"].duplicated().any():
    raise ValueError("Ada foto yang digunakan lebih dari sekali.")

if set(split_df["image_id"]) != set(image_ids):
    raise ValueError("Manifest dan embedding tidak cocok.")

id_lookup = {
    image_id: index
    for index, image_id in enumerate(image_ids)
}

mapped_labels = np.array([
    person_ids[id_lookup[value]]
    for value in split_df["image_id"]
])

if not np.array_equal(
    mapped_labels,
    split_df["person_id"].to_numpy()
):
    raise ValueError("Label manifest tidak sesuai embedding.")

gallery_df = split_df.loc[
    split_df["role"].eq("gallery")
]

gallery_labels = np.sort(gallery_df["person_id"].unique()).astype(str)

gallery_counts = gallery_df.groupby("person_id").size()

if len(gallery_labels) != len(known_ids) or not gallery_counts.eq(2).all():
    raise ValueError("Galeri harus mencakup seluruh known dengan dua foto per identitas.")

centroids = []

for person_id in gallery_labels:
    ids = gallery_df.loc[
        gallery_df["person_id"].eq(person_id),
        "image_id"
    ]

    indices = [id_lookup[value] for value in ids]

    centroid = X[indices].mean(axis=0)
    length = np.linalg.norm(centroid)

    if not np.isfinite(length) or length <= 1e-12:
        raise ValueError("Centroid tidak valid.")

    centroids.append(centroid / length)

centroids = np.stack(centroids).astype(np.float32)

gallery_path = ARTIFACTS_DIR / f"open_set_gallery_{len(gallery_labels)}.npz"

np.savez_compressed(
    gallery_path,
    centroids=centroids,
    person_ids=gallery_labels
)

print("Bentuk centroid:", centroids.shape)

## Skor validasi

In [ ]:
validation_df = split_df.loc[
    split_df["role"].isin([
        "validation_known",
        "validation_unknown"
    ])
].copy().reset_index(drop=True)

known_mask = validation_df["role"].eq("validation_known")
unknown_mask = validation_df["role"].eq("validation_unknown")

if not known_mask.any() or not unknown_mask.any():
    raise ValueError("Data validasi known dan unknown harus tersedia.")

if not validation_df.loc[
    known_mask, "person_id"
].isin(gallery_labels).all():
    raise ValueError("Known validasi tidak ditemukan dalam galeri.")

if validation_df.loc[
    unknown_mask, "person_id"
].isin(gallery_labels).any():
    raise ValueError("Identitas unknown ditemukan dalam galeri.")

indices = [
    id_lookup[value]
    for value in validation_df["image_id"]
]

score_matrix = X[indices] @ centroids.T
best_indices = np.argmax(score_matrix, axis=1)

validation_scores_df = validation_df.copy()

validation_scores_df["predicted_person_id"] = (
    gallery_labels[best_indices]
)

validation_scores_df["similarity"] = score_matrix[
    np.arange(len(validation_df)),
    best_indices
]

validation_scores_df["identity_correct"] = (
    validation_scores_df["person_id"]
    == validation_scores_df["predicted_person_id"]
)

REPORT_DIR = PROJECT_ROOT / "reports" / "experiments"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

validation_scores_df.to_csv(
    REPORT_DIR / "open_set_validation_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    validation_scores_df.groupby("role").agg(
        n_images=("image_id", "size"),
        min_similarity=("similarity", "min"),
        mean_similarity=("similarity", "mean"),
        max_similarity=("similarity", "max")
    ).round(4)
)

## Pemilihan threshold

In [ ]:
import json
import hashlib

scores = validation_scores_df["similarity"].to_numpy(
    dtype=np.float64
)

is_known = validation_scores_df["role"].eq(
    "validation_known"
).to_numpy()

identity_correct = (
    validation_scores_df["identity_correct"].to_numpy()
)

unique_scores = np.unique(scores)

# Titik tengah antar-skor mewakili perubahan keputusan.
candidates = [
    (float(unique_scores[0] - 1e-6), 0.0)
]

candidates.extend(
    (
        float((lower + upper) / 2),
        float(upper - lower)
    )
    for lower, upper in zip(
        unique_scores[:-1],
        unique_scores[1:]
    )
)

candidates.append(
    (float(unique_scores[-1] + 1e-6), 0.0)
)

threshold_rows = []

for threshold, gap_width in candidates:
    accepted = scores >= threshold

    known_correct_accepted = (
        is_known & identity_correct & accepted
    ).sum()

    unknown_accepted = (
        ~is_known & accepted
    ).sum()

    threshold_rows.append({
        "threshold": threshold,
        "score_gap_width": gap_width,
        "known_correct_accepted": int(known_correct_accepted),
        "known_correct_accept_rate": float(
            known_correct_accepted / is_known.sum()
        ),
        "known_rejected": int(
            (is_known & ~accepted).sum()
        ),
        "known_wrong_accepted": int(
            (is_known & ~identity_correct & accepted).sum()
        ),
        "unknown_accepted": int(unknown_accepted),
        "unknown_false_accept_rate": float(
            unknown_accepted / (~is_known).sum()
        ),
    })

threshold_table_df = pd.DataFrame(threshold_rows)

eligible_df = threshold_table_df.loc[
    threshold_table_df["unknown_accepted"].eq(0)
]

selected = (
    eligible_df.sort_values(
        [
            "known_correct_accept_rate",
            "score_gap_width",
            "threshold"
        ],
        ascending=[False, False, True]
    )
    .iloc[0]
)

selected_threshold = float(selected["threshold"])

threshold_table_df.to_csv(
    REPORT_DIR / "open_set_validation_thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)

config = {
    "method": "centroid",
    "threshold": selected_threshold,
    "accept_rule": "similarity >= threshold",
    "gallery_identities": len(gallery_labels),
    "gallery_filename": gallery_path.name,
    "gallery_photos_per_identity": 2,
    "validation_known_images": int(is_known.sum()),
    "validation_unknown_images": int((~is_known).sum()),
    "selection_policy": (
        "Zero observed unknown accepts; maximize correct "
        "known accepts; break ties by largest score gap."
    ),
    "gallery_sha256": hashlib.sha256(
        gallery_path.read_bytes()
    ).hexdigest(),
    "split_sha256": hashlib.sha256(
        split_path.read_bytes()
    ).hexdigest(),
    "embeddings_sha256": hashlib.sha256(
        embedding_path.read_bytes()
    ).hexdigest(),
}

config_path = ARTIFACTS_DIR / "open_set_threshold.json"

config_path.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print(f"Threshold terpilih: {selected_threshold:.6f}")
print(
    "Known benar dan diterima:",
    int(selected["known_correct_accepted"]),
    "/", int(is_known.sum())
)
print("Known ditolak:", int(selected["known_rejected"]))
print(
    "Known diterima dengan identitas salah:",
    int(selected["known_wrong_accepted"])
)
print(
    "Unknown salah diterima:",
    int(selected["unknown_accepted"]),
    "/", int((~is_known).sum())
)
print("Konfigurasi tersimpan:", config_path)

## Verifikasi sebelum test

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd

config_path = ARTIFACTS_DIR / "open_set_threshold.json"
embedding_path = ARTIFACTS_DIR / "arcface_embeddings.npz"
split_path = METADATA_DIR / "open_set_manifest.csv"

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

gallery_path = ARTIFACTS_DIR / config["gallery_filename"]

# Pastikan galeri, pembagian, dan embedding tidak berubah
# sejak threshold ditentukan.
for path, key in [
    (gallery_path, "gallery_sha256"),
    (split_path, "split_sha256"),
    (embedding_path, "embeddings_sha256"),
]:
    current_hash = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    if current_hash != config[key]:
        raise ValueError(
            f"File berubah sejak kalibrasi: {path.name}"
        )

threshold = float(config["threshold"])

# Baca centroid numerik dari file galeri.
with np.load(gallery_path, allow_pickle=False) as saved:
    test_centroids = saved["centroids"].copy()

# Ambil label sesuai urutan pembuatan galeri sebelumnya.
_gallery_manifest = pd.read_csv(split_path)

_gallery_rows = _gallery_manifest.loc[
    _gallery_manifest["role"].eq("gallery")
]

test_gallery_labels = np.asarray(
    sorted(_gallery_rows["person_id"].unique()),
    dtype=str
)

# Periksa pasangan label dan centroid menggunakan embedding sumber.
with np.load(embedding_path, allow_pickle=False) as saved:
    _source_X = saved["embeddings"]
    _source_ids = saved["image_ids"]
    _source_labels = saved["person_ids"]

_source_lookup = {
    image_id: index
    for index, image_id in enumerate(_source_ids)
}

_rebuilt_centroids = []

for person_id in test_gallery_labels:
    _ids = _gallery_rows.loc[
        _gallery_rows["person_id"].eq(person_id),
        "image_id"
    ]

    _indices = [_source_lookup[value] for value in _ids]

    if (
        len(_indices) != 2
        or not np.all(_source_labels[_indices] == person_id)
    ):
        raise ValueError("Referensi galeri tidak sesuai identitas.")

    _centroid = _source_X[_indices].mean(axis=0)
    _centroid = _centroid / np.linalg.norm(_centroid)

    _rebuilt_centroids.append(_centroid)

_rebuilt_centroids = np.stack(
    _rebuilt_centroids
).astype(np.float32)

if not np.array_equal(_rebuilt_centroids, test_centroids):
    raise ValueError(
        "Centroid tersimpan tidak cocok dengan urutan label manifest."
    )

print("Label dan centroid galeri berhasil diverifikasi.")
print("Jumlah identitas:", len(test_gallery_labels))

with np.load(embedding_path, allow_pickle=False) as saved:
    test_X = saved["embeddings"]
    test_image_ids = saved["image_ids"]
    test_person_ids = saved["person_ids"]

test_id_lookup = {
    image_id: index
    for index, image_id in enumerate(test_image_ids)
}

test_manifest_df = pd.read_csv(split_path)

test_df = test_manifest_df.loc[
    test_manifest_df["role"].isin([
        "test_known",
        "test_unknown"
    ])
].copy().reset_index(drop=True)

validation_unknown_ids = set(
    test_manifest_df.loc[
        test_manifest_df["role"].eq("validation_unknown"),
        "person_id"
    ]
)

test_unknown_ids = set(
    test_df.loc[
        test_df["role"].eq("test_unknown"),
        "person_id"
    ]
)

if validation_unknown_ids & test_unknown_ids:
    raise ValueError("Identitas unknown validasi dan test bertumpuk.")

print(f"Threshold tersimpan: {threshold:.8f}")
print(test_df["role"].value_counts().to_string())

## Prediksi test

In [ ]:
test_indices = np.array([
    test_id_lookup[value]
    for value in test_df["image_id"]
])

if not np.array_equal(
    test_person_ids[test_indices],
    test_df["person_id"].to_numpy()
):
    raise ValueError("Label foto test tidak sesuai.")

is_known = test_df["role"].eq("test_known").to_numpy()

if not test_df.loc[
    is_known, "person_id"
].isin(test_gallery_labels).all():
    raise ValueError("Identitas known tidak ditemukan dalam galeri.")

if test_df.loc[
    ~is_known, "person_id"
].isin(test_gallery_labels).any():
    raise ValueError("Identitas unknown ditemukan dalam galeri.")

scores = test_X[test_indices] @ test_centroids.T
best_indices = np.argmax(scores, axis=1)

candidates = test_gallery_labels[best_indices]

best_scores = scores[
    np.arange(len(test_df)),
    best_indices
]

accepted = best_scores >= threshold

identity_correct = (
    candidates == test_df["person_id"].to_numpy()
)

test_df["candidate_person_id"] = candidates
test_df["similarity"] = best_scores
test_df["accepted"] = accepted

test_df["final_prediction"] = np.where(
    accepted, candidates, "UNKNOWN"
)

test_df["outcome"] = np.select(
    [
        is_known & accepted & identity_correct,
        is_known & ~accepted,
        is_known & accepted & ~identity_correct,
        ~is_known & ~accepted,
        ~is_known & accepted,
    ],
    [
        "known_correct_accepted",
        "known_rejected",
        "known_wrong_accepted",
        "unknown_correct_rejected",
        "unknown_wrong_accepted",
    ],
    default="invalid"
)

REPORT_DIR = PROJECT_ROOT / "reports" / "experiments"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

test_df.to_csv(
    REPORT_DIR / "open_set_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluasi selesai:", len(test_df), "foto.")

## Ringkasan test

In [ ]:
outcome_order = [
    "known_correct_accepted",
    "known_rejected",
    "known_wrong_accepted",
    "unknown_correct_rejected",
    "unknown_wrong_accepted",
]

counts = (
    test_df["outcome"]
    .value_counts()
    .reindex(outcome_order, fill_value=0)
)

display(
    counts.rename_axis("outcome")
    .reset_index(name="jumlah_foto")
)

n_known = int(is_known.sum())
n_unknown = int((~is_known).sum())

known_success_rate = (
    int(counts["known_correct_accepted"]) / n_known
)

unknown_false_accept_rate = (
    int(counts["unknown_wrong_accepted"]) / n_unknown
)

unknown_rejection_rate = (
    int(counts["unknown_correct_rejected"]) / n_unknown
)

summary = {
    "threshold": float(threshold),
    "n_known": n_known,
    "n_unknown": n_unknown,
    "counts": {
        key: int(value)
        for key, value in counts.items()
    },
    "known_correct_accept_rate": known_success_rate,
    "unknown_false_accept_rate": unknown_false_accept_rate,
    "unknown_rejection_rate": unknown_rejection_rate,
    "balanced_open_set_accuracy": (
        known_success_rate + unknown_rejection_rate
    ) / 2,
}

(REPORT_DIR / "open_set_test_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print(f"Known benar dan diterima : {known_success_rate:.2%}")
print(f"Unknown salah diterima   : {unknown_false_accept_rate:.2%}")
print(f"Unknown berhasil ditolak : {unknown_rejection_rate:.2%}")

print("\nKasus yang gagal:")

display(
    test_df.loc[
        ~test_df["outcome"].isin([
            "known_correct_accepted",
            "unknown_correct_rejected"
        ]),
        [
            "image_id",
            "person_id",
            "candidate_person_id",
            "similarity",
            "final_prediction",
            "outcome"
        ]
    ]
)